# Part 1: Data Transformation

This notebook Transform raw data from PostgreSQL, MongoDB, and the Open-Meteo weather API into a clean, consistent datasets that we can later load into the data warehouse.

In [7]:
import json
from pathlib import Path

import pandas as pd
import numpy as np

# Find project root
current_dir = Path.cwd()

if (current_dir / ".env").exists():
    PROJECT_DIR = current_dir
else:
    PROJECT_DIR = current_dir.parent

RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)

Raw folder: d:\Atomcamp AI Bootcamp\6. ETL Module\Online-Shop-ETL-Project\data\raw
Processed folder: d:\Atomcamp AI Bootcamp\6. ETL Module\Online-Shop-ETL-Project\data\processed


In [17]:
# Load PostgreSQL datasets (customers and orders)

customers_df = pd.read_csv(
    RAW_DIR / "postgres_customers.csv"
)

orders_df = pd.read_csv(
    RAW_DIR / "postgres_orders.csv"
)

print("Customers:", customers_df.shape)
print("PostgreSQL orders:", orders_df.shape)

display(customers_df.head())
display(orders_df.head())

Customers: (2000, 5)
PostgreSQL orders: (200000, 9)


,customer_id,name,city,segment,signup_date
0,1,Usman Sheikh,Karachi,Consumer,2022-11-28
1,2,Iqra Khan,Faisalabad,Consumer,2022-11-11
2,3,Maryam Raza,Faisalabad,Consumer,2023-10-20
3,4,Ali Sheikh,Lahore,Consumer,2022-08-09
4,5,Hamza Qureshi,Karachi,Consumer,2022-11-10


,order_id,customer_id,order_date,category,quantity,amount,status,payment_method,updated_at
0,100001,261,2026-09-01,Electronics,1,77635.24,delivered,bank_transfer,2026-09-07 14:58:52
1,100002,101,2025-02-28,Grocery,1,5562.63,delivered,card,2025-03-05 07:26:15
2,100003,777,2026-07-08,Books,1,2055.15,shipped,card,2026-07-09 23:35:51
3,100004,1509,2025-02-22,Beauty,1,7414.43,delivered,wallet,2025-03-01 21:29:06
4,100005,162,2026-01-12,Books,4,6784.53,delivered,wallet,2026-01-14 02:59:49


In [9]:
# Load MongoDB JSON documents

with open(
    RAW_DIR / "mongo_orders.json",
    "r",
    encoding="utf-8"
) as file:

    mongo_documents = json.load(file)

print("MongoDB documents:", len(mongo_documents))

# Display one original nested document
mongo_documents[0]

MongoDB documents: 5000


{'order_id': 238188,
 'order_date': '2025-01-01T02:57:21',
 'status': 'shipped',
 'amount': 3153.9,
 'items': [{'category': 'Books', 'quantity': 1}],
 'payment': {'method': 'card', 'installments': 3},
 'updated_at': '2025-01-04T05:39:21',
 'customer': {'id': 304, 'name': 'Imran Sheikh', 'city': 'Sialkot'}}

### Clean the MongoDB Data

In [10]:
# Flatten MongoDB documents

mongo_rows = []

for doc in mongo_documents:

    # Extract nested customer information
    customer = doc.get("customer") or {}

    # Extract first item from the items list
    items = doc.get("items") or []
    item = items[0] if items else {}

    # Extract nested payment information
    payment = doc.get("payment") or {}

    # Get customer ID from either location
    customer_id = customer.get("id")

    if customer_id is None:
        customer_id = doc.get("customer_id")

    row = {
        "order_id": doc.get("order_id"),
        "customer_id": customer_id,
        "order_date": doc.get("order_date"),
        "category": item.get("category"),
        "quantity": item.get("quantity"),
        "amount": doc.get("amount"),
        "status": doc.get("status"),
        "payment_method": payment.get("method"),
        "updated_at": doc.get("updated_at"),

        # Additional app customer information
        "customer_city": customer.get("city"),
        "customer_name": customer.get("name"),
        "customer_segment": customer.get("segment"),

        # Optional fields
        "coupon": doc.get("coupon"),
        "delivery_notes": doc.get("delivery_notes"),
        "installments": payment.get("installments"),

        "source": "app"
    }

    mongo_rows.append(row)

mongo_orders_df = pd.DataFrame(mongo_rows)

print("Flattened MongoDB orders:", mongo_orders_df.shape)

display(mongo_orders_df.head())

Flattened MongoDB orders: (5000, 16)


,order_id,customer_id,order_date,category,quantity,amount,status,payment_method,updated_at,customer_city,customer_name,customer_segment,coupon,delivery_notes,installments,source
0,238188,304,2025-01-01T02:57:21,Books,1,3153.90,shipped,card,2025-01-04T05:39:21,Sialkot,Imran Sheikh,NaN,NaN,NaN,3.0,app
1,110570,499,2025-01-01T05:06:31,Home,3,30188.21,delivered,wallet,2025-01-03T08:03:31,NaN,NaN,NaN,NaN,NaN,NaN,app
2,237480,1750,2025-01-01T05:14:21,Fashion,1,14310.06,pending,cash_on_delivery,2025-01-01T07:23:21,Karachi,Ali Raza,NaN,EID25,Office hours only,NaN,app
3,225926,306,2025-01-01T10:09:15,Electronics,1,55997.47,pending,bank_transfer,2025-01-01T11:49:15,NaN,NaN,NaN,NaN,NaN,6.0,app
4,267865,104,2025-01-01T13:59:25,Electronics,1,18810.34,delivered,cash_on_delivery,2025-01-04T16:01:25,Karachi,Noor Khan,NaN,NaN,NaN,NaN,app


In [11]:
# Save flattened MongoDB data
mongo_orders_df.to_csv(
    PROCESSED_DIR / "mongo_orders_flat.csv",
    index=False
)

print("Flattened MongoDB data saved!")


Flattened MongoDB data saved!


### Clean the PostgreSQL data

In [ ]:
# PostgreSQL customers quality overview

print("CUSTOMERS")
print("Missing values:")
print(customers_df.isnull().sum())

print("\nDuplicate customer IDs:")
print(customers_df["customer_id"].duplicated().sum())


# PostgreSQL orders quality overview

print("\nPOSTGRESQL ORDERS")
print("Missing values:")
print(orders_df.isnull().sum())

print("\nDuplicate order IDs:")
print(orders_df["order_id"].duplicated().sum())

CUSTOMERS
Missing values:
customer_id    0
name           0
city           0
segment        0
signup_date    0
dtype: int64

Duplicate customer IDs:
0

POSTGRESQL ORDERS
Missing values:
order_id          0
customer_id       0
order_date        0
category          0
quantity          0
amount            0
status            0
payment_method    0
updated_at        0
dtype: int64

Duplicate order IDs:
0


In [12]:
# Standardize PostgreSQL orders

# Make a working copy
web_orders_df = orders_df.copy()

# Mark PostgreSQL orders as web orders
web_orders_df["source"] = "web"

# Add columns that exist in MongoDB but not PostgreSQL
extra_columns = [
    "customer_city",
    "customer_name",
    "customer_segment",
    "coupon",
    "delivery_notes",
    "installments"
]

for column in extra_columns:
    web_orders_df[column] = np.nan

print("PostgreSQL orders standardized.")

display(web_orders_df.head())

PostgreSQL orders standardized.


,order_id,customer_id,order_date,category,quantity,amount,status,payment_method,updated_at,source,customer_city,customer_name,customer_segment,coupon,delivery_notes,installments
0,100001,261,2026-09-01,Electronics,1,77635.24,delivered,bank_transfer,2026-09-07 14:58:52,web,NaN,NaN,NaN,NaN,NaN,NaN
1,100002,101,2025-02-28,Grocery,1,5562.63,delivered,card,2025-03-05 07:26:15,web,NaN,NaN,NaN,NaN,NaN,NaN
2,100003,777,2026-07-08,Books,1,2055.15,shipped,card,2026-07-09 23:35:51,web,NaN,NaN,NaN,NaN,NaN,NaN
3,100004,1509,2025-02-22,Beauty,1,7414.43,delivered,wallet,2025-03-01 21:29:06,web,NaN,NaN,NaN,NaN,NaN,NaN
4,100005,162,2026-01-12,Books,4,6784.53,delivered,wallet,2026-01-14 02:59:49,web,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
# Standardize MongoDB orders

# Keep the same columns and column order in both sources
standard_columns = [
    "order_id",
    "customer_id",
    "order_date",
    "category",
    "quantity",
    "amount",
    "status",
    "payment_method",
    "updated_at",
    "customer_city",
    "customer_name",
    "customer_segment",
    "coupon",
    "delivery_notes",
    "installments",
    "source"
]

web_orders_df = web_orders_df[standard_columns].copy()
mongo_orders_df = mongo_orders_df[standard_columns].copy()

print("Web columns:", len(web_orders_df.columns))
print("App columns:", len(mongo_orders_df.columns))

print(
    "Both sources have identical columns:",
    list(web_orders_df.columns) == list(mongo_orders_df.columns)
)

Web columns: 16
App columns: 16
Both sources have identical columns: True


In [14]:
# Combine web and app orders

all_orders_df = pd.concat(
    [web_orders_df, mongo_orders_df],
    ignore_index=True
)

print("Combined orders:", len(all_orders_df))

print("\nRecords by source:")
print(all_orders_df["source"].value_counts())

display(all_orders_df.head())

Combined orders: 205000

Records by source:
source
web    200000
app      5000
Name: count, dtype: int64


,order_id,customer_id,order_date,category,quantity,amount,status,payment_method,updated_at,customer_city,customer_name,customer_segment,coupon,delivery_notes,installments,source
0,100001,261,2026-09-01,Electronics,1,77635.24,delivered,bank_transfer,2026-09-07 14:58:52,NaN,NaN,NaN,NaN,NaN,NaN,web
1,100002,101,2025-02-28,Grocery,1,5562.63,delivered,card,2025-03-05 07:26:15,NaN,NaN,NaN,NaN,NaN,NaN,web
2,100003,777,2026-07-08,Books,1,2055.15,shipped,card,2026-07-09 23:35:51,NaN,NaN,NaN,NaN,NaN,NaN,web
3,100004,1509,2025-02-22,Beauty,1,7414.43,delivered,wallet,2025-03-01 21:29:06,NaN,NaN,NaN,NaN,NaN,NaN,web
4,100005,162,2026-01-12,Books,4,6784.53,delivered,wallet,2026-01-14 02:59:49,NaN,NaN,NaN,NaN,NaN,NaN,web


In [15]:
# Identify duplicate orders

duplicate_count = all_orders_df.duplicated(
    subset=["order_id"]
).sum()

print("Duplicate order records:", duplicate_count)

print(
    "Unique order IDs before deduplication:",
    all_orders_df["order_id"].nunique()
)

Duplicate order records: 5000
Unique order IDs before deduplication: 200000


In [18]:
# Handle duplicates

# Combine source labels for each order
def combine_sources(values):
    sources = sorted(set(values.dropna()))
    return "+".join(sources)


# Prefer the web row when choosing the main order record.
# The groupby first() operation also fills missing values
# from another record when available.
all_orders_df["source_priority"] = (
    all_orders_df["source"].map({
        "web": 0,
        "app": 1
    })
)

all_orders_df = all_orders_df.sort_values(
    ["order_id", "source_priority"]
)

aggregation = {
    column: "first"
    for column in standard_columns
    if column not in ["order_id", "source"]
}

aggregation["source"] = combine_sources

orders_clean = (
    all_orders_df
    .groupby("order_id", as_index=False)
    .agg(aggregation)
)

print("Orders after deduplication:", len(orders_clean))
print("Unique order IDs:", orders_clean["order_id"].nunique())

display(orders_clean.head())

Orders after deduplication: 200000
Unique order IDs: 200000


,order_id,customer_id,order_date,category,quantity,amount,status,payment_method,updated_at,customer_city,customer_name,customer_segment,coupon,delivery_notes,installments,source
0,100001,261,2026-09-01,Electronics,1,77635.24,delivered,bank_transfer,2026-09-07 14:58:52,None,None,None,None,None,NaN,web
1,100002,101,2025-02-28,Grocery,1,5562.63,delivered,card,2025-03-05 07:26:15,None,None,None,None,None,NaN,web
2,100003,777,2026-07-08,Books,1,2055.15,shipped,card,2026-07-09 23:35:51,None,None,None,None,None,NaN,web
3,100004,1509,2025-02-22,Beauty,1,7414.43,delivered,wallet,2025-03-01 21:29:06,None,None,None,None,None,NaN,web
4,100005,162,2026-01-12,Books,4,6784.53,delivered,wallet,2026-01-14 02:59:49,None,None,None,None,None,NaN,web


In [26]:
# Fix data types

# Convert IDs and quantities to numeric types
orders_clean["order_id"] = pd.to_numeric(
    orders_clean["order_id"],
    errors="coerce"
).astype("Int64")

orders_clean["customer_id"] = pd.to_numeric(
    orders_clean["customer_id"],
    errors="coerce"
).astype("Int64")

orders_clean["quantity"] = pd.to_numeric(
    orders_clean["quantity"],
    errors="coerce"
).astype("Int64")

orders_clean["amount"] = pd.to_numeric(
    orders_clean["amount"],
    errors="coerce"
)

orders_clean["installments"] = pd.to_numeric(
    orders_clean["installments"],
    errors="coerce"
).astype("Int64")

# Convert date columns
orders_clean["order_date"] = pd.to_datetime(
    orders_clean["order_date"],
    format="mixed",
    errors="coerce"
).dt.normalize()

orders_clean["updated_at"] = pd.to_datetime(
    orders_clean["updated_at"],
    format="mixed",
    errors="coerce"
)

# Clean text columns
text_columns = [
    "category",
    "status",
    "payment_method",
    "customer_city",
    "customer_name",
    "customer_segment",
    "coupon",
    "delivery_notes",
    "source"
]

for column in text_columns:
    orders_clean[column] = (
        orders_clean[column]
        .astype("string")
        .str.strip()
    )

print("Data types standardized.")

display(orders_clean.dtypes)



Data types standardized.


order_id                            Int64
customer_id                         Int64
order_date                 datetime64[us]
category                           string
quantity                            Int64
amount                            float64
status                             string
payment_method                     string
updated_at                 datetime64[us]
customer_city                      string
customer_name                      string
customer_segment                   string
coupon                             string
delivery_notes                     string
installments                        Int64
source                             string
master_customer_name                  str
master_customer_city                  str
master_customer_segment               str
signup_date                           str
dtype: object

In [20]:
# Enrich orders with customer information

# Standardize customer IDs
customers_df["customer_id"] = pd.to_numeric(
    customers_df["customer_id"],
    errors="coerce"
).astype("Int64")

# Prepare customer lookup
customer_lookup = customers_df[
    ["customer_id", "name", "city", "segment", "signup_date"]
].copy()

customer_lookup = customer_lookup.rename(columns={
    "name": "master_customer_name",
    "city": "master_customer_city",
    "segment": "master_customer_segment"
})

# Join customer information to orders
orders_clean = orders_clean.merge(
    customer_lookup,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

# Fill missing app customer information from PostgreSQL
orders_clean["customer_city"] = (
    orders_clean["customer_city"]
    .fillna(orders_clean["master_customer_city"])
)

orders_clean["customer_name"] = (
    orders_clean["customer_name"]
    .fillna(orders_clean["master_customer_name"])
)

orders_clean["customer_segment"] = (
    orders_clean["customer_segment"]
    .fillna(orders_clean["master_customer_segment"])
)

print("Customer information enriched.")

display(
    orders_clean[
        ["order_id", "customer_id", "customer_city", "customer_name"]
    ].head()
)

#Some app documents do not contain customer names or cities. We'll use the PostgreSQL customers table to fill missing customer details.

Customer information enriched.


,order_id,customer_id,customer_city,customer_name
0,100001,261,Faisalabad,Ali Khan
1,100002,101,Rawalpindi,Ali Abbasi
2,100003,777,Rawalpindi,Usman Malik
3,100004,1509,Lahore,Usman Butt
4,100005,162,Peshawar,Bilal Malik


In [21]:
# Run data quality checks

valid_statuses = {
    "delivered",
    "shipped",
    "pending",
    "cancelled",
    "returned"
}

customer_ids = set(
    customers_df["customer_id"].dropna().astype(int)
)

order_customer_ids = set(
    orders_clean["customer_id"].dropna().astype(int)
)

quality_checks = {
    "No missing order IDs": orders_clean["order_id"].notna().all(),

    "No duplicate order IDs": (
        orders_clean["order_id"].is_unique
    ),

    "No missing customer IDs": (
        orders_clean["customer_id"].notna().all()
    ),

    "All customer IDs exist in customer table": (
        order_customer_ids.issubset(customer_ids)
    ),

    "All order amounts are positive": (
        orders_clean["amount"].notna().all()
        and (orders_clean["amount"] > 0).all()
    ),

    "All quantities are positive": (
        orders_clean["quantity"].notna().all()
        and (orders_clean["quantity"] > 0).all()
    ),

    "All order statuses are valid": (
        orders_clean["status"].isin(valid_statuses).all()
    ),

    "All order dates are valid": (
        orders_clean["order_date"].notna().all()
    ),

    "All source labels are valid": (
        orders_clean["source"].isin(
            ["web", "app", "app+web"]
        ).all()
    )
}

for check_name, passed in quality_checks.items():

    status = "PASS" if passed else "FAIL"

    print(f"{status}: {check_name}")

PASS: No missing order IDs
PASS: No duplicate order IDs
PASS: No missing customer IDs
PASS: All customer IDs exist in customer table
PASS: All order amounts are positive
PASS: All quantities are positive
PASS: All order statuses are valid
PASS: All order dates are valid
PASS: All source labels are valid


### Clean the weather data

In [22]:
weather_df = pd.read_csv(
    RAW_DIR / "weather_daily.csv"
)

# Standardize weather date
weather_df["date"] = pd.to_datetime(
    weather_df["date"],
    errors="coerce"
).dt.normalize()

# Standardize city names
weather_df["city"] = (
    weather_df["city"]
    .astype("string")
    .str.strip()
)

# Convert weather measurements to numeric
weather_df["temperature_2m_max"] = pd.to_numeric(
    weather_df["temperature_2m_max"],
    errors="coerce"
)

weather_df["precipitation_sum"] = pd.to_numeric(
    weather_df["precipitation_sum"],
    errors="coerce"
)

print("Weather data cleaned:", weather_df.shape)

display(weather_df.head())

Weather data cleaned: (1986, 4)


,date,temperature_2m_max,precipitation_sum,city
0,2025-01-01,16.3,0.0,Lahore
1,2025-01-02,17.2,0.0,Lahore
2,2025-01-03,17.4,0.0,Lahore
3,2025-01-04,18.8,0.0,Lahore
4,2025-01-05,15.9,36.6,Lahore


In [23]:
# Join weather data with orders

orders_with_weather = orders_clean.merge(
    weather_df,
    left_on=["customer_city", "order_date"],
    right_on=["city", "date"],
    how="left",
    validate="many_to_one"
)

# Remove duplicate join columns
orders_with_weather = orders_with_weather.drop(
    columns=["city", "date"]
)

print("Orders with weather:", orders_with_weather.shape)

weather_matched = orders_with_weather[
    "temperature_2m_max"
].notna().sum()

print("Orders matched with weather:", weather_matched)
print(
    "Orders without weather match:",
    len(orders_with_weather) - weather_matched
)

display(orders_with_weather.head())

# used customer city and order date to match an order with the weather on that date in that city.

Orders with weather: (200000, 22)
Orders matched with weather: 117364
Orders without weather match: 82636


,order_id,customer_id,order_date,category,quantity,amount,status,payment_method,updated_at,customer_city,...,coupon,delivery_notes,installments,source,master_customer_name,master_customer_city,master_customer_segment,signup_date,temperature_2m_max,precipitation_sum
0,100001,261,2026-09-01,Electronics,1,77635.24,delivered,bank_transfer,2026-09-07 14:58:52,Faisalabad,...,<NA>,<NA>,<NA>,web,Ali Khan,Faisalabad,Consumer,2023-04-18,NaN,NaN
1,100002,101,2025-02-28,Grocery,1,5562.63,delivered,card,2025-03-05 07:26:15,Rawalpindi,...,<NA>,<NA>,<NA>,web,Ali Abbasi,Rawalpindi,Consumer,2023-07-20,NaN,NaN
2,100003,777,2026-07-08,Books,1,2055.15,shipped,card,2026-07-09 23:35:51,Rawalpindi,...,<NA>,<NA>,<NA>,web,Usman Malik,Rawalpindi,Corporate,2024-08-04,NaN,NaN
3,100004,1509,2025-02-22,Beauty,1,7414.43,delivered,wallet,2025-03-01 21:29:06,Lahore,...,<NA>,<NA>,<NA>,web,Usman Butt,Lahore,Consumer,2024-04-15,21.9,0.0
4,100005,162,2026-01-12,Books,4,6784.53,delivered,wallet,2026-01-14 02:59:49,Peshawar,...,<NA>,<NA>,<NA>,web,Bilal Malik,Peshawar,Consumer,2022-10-05,NaN,NaN


In [24]:
# Save transformed data

# Save cleaned customers
customers_df.to_csv(
    PROCESSED_DIR / "customers_clean.csv",
    index=False
)

# Save cleaned and deduplicated orders
orders_clean.to_csv(
    PROCESSED_DIR / "orders_clean.csv",
    index=False
)

# Save orders enriched with weather
orders_with_weather.to_csv(
    PROCESSED_DIR / "orders_with_weather.csv",
    index=False
)

# Save cleaned weather
weather_df.to_csv(
    PROCESSED_DIR / "weather_clean.csv",
    index=False
)

print("All transformed datasets saved successfully!")

All transformed datasets saved successfully!


In [25]:
print("========== TRANSFORMATION SUMMARY ==========")

print("Customers:", len(customers_df))
print("PostgreSQL orders:", len(web_orders_df))
print("MongoDB orders:", len(mongo_orders_df))
print("Combined records:", len(all_orders_df))
print("Unique cleaned orders:", len(orders_clean))
print("Orders with weather:", len(orders_with_weather))
print("Weather records:", len(weather_df))

print("\n========== QUALITY CHECK SUMMARY ==========")

passed_checks = sum(quality_checks.values())
total_checks = len(quality_checks)

print(f"Passed: {passed_checks}/{total_checks}")

if passed_checks != total_checks:
    print("Please investigate the failed checks before proceeding.")
else:
    print("All quality checks passed!")

print("\nTransformation completed!")

========== TRANSFORMATION SUMMARY ==========
Customers: 2000
PostgreSQL orders: 200000
MongoDB orders: 5000
Combined records: 205000
Unique cleaned orders: 200000
Orders with weather: 200000
Weather records: 1986

========== QUALITY CHECK SUMMARY ==========
Passed: 9/9
All quality checks passed!

Transformation completed!
